In [6]:
# imports
import pandas as pd
import numpy as np
import yfinance as yf
from pandas_datareader import data as pdr

In [7]:
# Define the FX tickers and their corresponding FRED rates

FX_TICKERS = {
    "AUD": {"ticker": "AUDUSD=X", "invert": False},
    "CAD": {"ticker": "CADUSD=X", "invert": True}, 
    "CHF": {"ticker": "CHFUSD=X", "invert": True},
    "EUR": {"ticker": "EURUSD=X", "invert": False},
    "GBP": {"ticker": "GBPUSD=X", "invert": False},
    "JPY": {"ticker": "JPYUSD=X", "invert": True}, 
    "NZD": {"ticker": "NZDUSD=X", "invert": False}
}

FRED_RATES = {
    "USD": "DFF",   
    "AUD": "IRSTCI01AUM156N", 
    "CAD": "IRSTCI01CAM156N",
    "CHF": "IRSTCI01CHM156N",
    "EUR": "IRSTCI01EZM156N",
    "GBP": "IRSTCI01GBM156N",
    "JPY": "IRSTCI01JPM156N",
    "NZD": "IRSTCI01NZM156N"
}

In [8]:
# FX PRICE DATA (t is close price, usable at t+1)
def get_fx_data(ticker, start="2000-01-01", end=None):
    df = yf.download(ticker, start=start, end=end, progress=False)

    if df.empty:
        raise ValueError(f"No data for {ticker}")

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    return df[["Close"]].rename(columns={"Close": "price"}).dropna()


def build_fx_price_panel(fx_map, start="2000-01-01", end=None):
    out = {}

    for ccy, meta in fx_map.items():
        df = get_fx_data(meta["ticker"], start, end)
        price = df["price"].copy()

        if meta["invert"]:
            price = 1.0 / price

        out[f"{ccy}_price"] = price

    panel = pd.concat(out.values(), axis=1)
    panel.columns = out.keys()

    return panel.sort_index()

# RATE DATA (macro, must be lagged before use)
def get_fred_rate(series, start="2000-01-01", end=None):
    df = pdr.DataReader(series, "fred", start, end)
    return df.rename(columns={series: "rate"}) / 100.0


def build_rate_panel(rate_map, start="2000-01-01", end=None):
    out = {}

    for ccy, series in rate_map.items():
        df = get_fred_rate(series, start, end)
        df.columns = [f"{ccy}_rate"]
        out[ccy] = df

    panel = pd.concat(out.values(), axis=1).sort_index()

    # DO NOT ffill yet — do it explicitly post-lag
    panel = panel.loc[panel["USD_rate"].first_valid_index():]

    return panel

# AS-OF ENFORCEMENT
def apply_rate_lag_and_fill(rate_df, target_index):
    """
    Enforce as-of timing:
    - Lag all macro/rates by 1 observation
    - Reindex to daily FX price calendar
    - Forward fill after lag
    """

    rates_lagged = rate_df.shift(1)
    rates_lagged = rates_lagged.reindex(target_index).ffill()

    return rates_lagged

# SYSTEM ASSEMBLY
def build_fx_system(fx_map, rate_map, start="2000-01-01", end=None):

    price_df = build_fx_price_panel(fx_map, start, end)
    rate_df_raw = build_rate_panel(rate_map, start, end)

    # enforce lag and align rates to daily FX calendar
    rate_df = apply_rate_lag_and_fill(rate_df_raw, price_df.index)

    df = pd.concat([price_df, rate_df], axis=1)

    return df.sort_index(), rate_df_raw, rate_df

# FX RETURNS COMPUTATION (uses lagged rates)
def compute_fx_returns(df, fx_map):

    out = pd.DataFrame(index=df.index)

    for ccy in fx_map.keys():
        price = df[f"{ccy}_price"]
        r_spot = np.log(price).diff()

        r_usd = df["USD_rate"]
        r_fx = df[f"{ccy}_rate"]

        # carry uses lagged rates (already enforced upstream)
        r_carry = (r_fx - r_usd) / 252

        out[ccy] = r_spot + r_carry

    return out

In [9]:
# DATA AUDIT CHECKS
def audit_data(df):
    audit = {}

    # % missing by column
    audit["missing_pct"] = df.isna().mean()

    # first valid date per series
    audit["first_valid"] = df.apply(lambda x: x.first_valid_index())

    # forward fill length distribution
    ffill_lengths = {}

    for col in df.columns:
        series = df[col]
        is_ffill = series.isna()

        lengths = []
        current = 0

        for val in is_ffill:
            if val:
                current += 1
            else:
                if current > 0:
                    lengths.append(current)
                current = 0

        if current > 0:
            lengths.append(current)

        ffill_lengths[col] = lengths

    audit["ffill_lengths"] = ffill_lengths

    return audit

In [10]:
# BUILD SYSTEM

df, rate_df_raw, rate_df_lagged = build_fx_system(
    FX_TICKERS, FRED_RATES, start="2000-01-01"
)

returns_df = compute_fx_returns(df, FX_TICKERS)

# ensure exact index alignment
common_idx = df.index.intersection(returns_df.index)
df = df.loc[common_idx].copy()
returns_df = returns_df.loc[common_idx].copy()

# drop initial incomplete rows
first_full = df.dropna().index.min()
df = df.loc[first_full:]
returns_df = returns_df.loc[first_full:]

# AUDIT OUTPUT
audit_df = audit_data(df)
audit_rates_raw = audit_data(rate_df_raw)
audit_rates_lagged = audit_data(rate_df_lagged)

# SAVE PARQUETS (EXPLICITLY LABELED AS LAGGED)
df.to_parquet("data/fx_system_df_lagged.parquet")
returns_df.to_parquet("data/fx_returns_df_lagged.parquet")

rate_df_raw.to_parquet("data/fx_rates_raw.parquet")
rate_df_lagged.to_parquet("data/fx_rates_lagged.parquet")